In [2]:
import pandas as pd
fgk = pd.read_csv("fgk.txt", sep="\t")

In [3]:
fgk.head()

,star,group,ID,ID_alt,star_alt1,star_alt2,origin,snr,R,Rmax,...,e[Ti 1/H],n[Ti 1/H],[Ti 2/H],e[Ti 2/H],n[Ti 2/H],[V 1/H],e[V 1/H],n[V 1/H],n_spectra,flag
0,HIP101345,G Subgiant (IV),HD195564_HAR_1,HIP101345_HARPS_1,HD195564,-,HARPS,902,42000,115000,...,0.023,34,0.062,0.027,7,-0.121,0.024,27,2,-
1,HIP101345,G Subgiant (IV),HD195564_NAR_1,HIP101345_NARVAL_1,HD195564,-,NARVAL,369,42000,68000,...,0.023,34,0.062,0.027,7,-0.121,0.024,27,2,-
2,HIP10234,K Giant (III),HD13468_FER_1,HIP10234_FEROS_1,HD13468,-,FEROS,121,42000,48000,...,0.059,36,-0.221,0.042,8,-0.532,0.061,29,2,-
3,HIP10234,K Giant (III),HD13468_HAR_1,HIP10234_HARPS_1,HD13468,-,HARPS,95,42000,115000,...,0.059,36,-0.221,0.042,8,-0.532,0.061,29,2,-
4,HIP102422,K Subgiant (IV),HD198149_NAR_1,HIP102422_NARVAL_1,HD198149,-,NARVAL,908,42000,68000,...,0.062,35,-0.218,0.031,8,-0.447,0.047,30,2,-


In [4]:
import pyvo

tap = pyvo.dal.TAPService("https://archive.eso.org/tap_obs")

# Query to get the columns of the ivoa.obscore table
query = """
SELECT column_name, datatype, description 
FROM TAP_SCHEMA.columns 
WHERE table_name = 'ivoa.obscore'
"""

result = tap.search(query)

for row in result:
    print(f"{row['column_name']}: {row['datatype']} - {row['description']}")

abmaglim: double - ESO-specific field not present in the standard ObsCore. 5-sigma limiting AB (Oke) magnitude. The quoted magnitude should refer to the total flux of a point source. Applicable to the following data product type: cube, image, and measurements.
access_estsize: long - Estimated size of the downloaded file in KBytes.
access_format: char - The format of the downloaded file.
access_url: char - A URL that points to the DataLink (VO Standard) service that is used to download the dataset, associated files, their provenance or derived products, etc.
bib_reference: char - URL or bibcode to the main publication that refers to the data set.
calib_level: int - Calibration level: 0-instrument (raw) data in non-standard format, 1-instrumental (raw) data in standard format, 2-science ready data with instrument signature removed, 3-more highly processed data. Ref. PRODLVL keyword in ESO SDP standard if present, otherwise: set to 3 when dataproduct_type contains token deep or tile, set 

In [16]:
import os
def read_eso_credentials():
    # First, try environment variables
    user = os.environ.get("ESO_USERNAME")
    pw = os.environ.get("ESO_PASSWORD")

    if user and pw:
        return user, pw

    # If not found, try eso.env file
    env_file = "eso.env"
    if os.path.exists(env_file):
        creds = {}
        with open(env_file) as f:
            for line in f:
                if "=" in line:
                    k, v = line.strip().split("=", 1)
                    creds[k.strip()] = v.strip()
        user = creds.get("ESO_USERNAME")
        pw = creds.get("ESO_PASSWORD")
        if user and pw:
            return user, pw

    raise EnvironmentError(
        "Please set ESO_USERNAME and ESO_PASSWORD environment variables "
        "or provide them in an eso.env file for authentication."
    )

In [19]:
import pyvo
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
import os

def download_data(instrument, target, output_dir):
    USER, PASS = read_eso_credentials()

    tap = pyvo.dal.TAPService("https://archive.eso.org/tap_obs")

    import re

    def safe_str(s):
        return re.sub(r"[^a-zA-Z0-9_\- ]", "", str(s))

    safe_instrument = safe_str(instrument)
    safe_target = safe_str(target)

    query = f"""
    SELECT obs_publisher_did, access_url
    FROM ivoa.obscore
    WHERE instrument_name='{safe_instrument}'
    AND target_name='{safe_target}'
    """

    result = tap.search(query)

    outdir = Path(os.path.join(output_dir, safe_instrument, safe_target))
    outdir.mkdir(parents=True, exist_ok=True)

    target_name = safe_target
    file_index = 0

    for row in result:
        datalink_url = row["access_url"]
        print("DATALINK:", datalink_url)

        # Step 1: fetch datalink XML
        try:
            dl = requests.get(datalink_url, auth=(USER, PASS))
            dl.raise_for_status()
        except requests.HTTPError as err:
            if err.response is not None and err.response.status_code == 401:
                print(f"401 Unauthorized Error while fetching DataLink: Check your ESO credentials. ({datalink_url})")
                break
            else:
                print(f"Failed to fetch DataLink: {err} ({datalink_url})")
                continue
        except Exception as err:
            print(f"Unexpected error fetching DataLink: {err} ({datalink_url})")
            continue

        # Step 2: parse XML
        try:
            root = ET.fromstring(dl.text)
        except ET.ParseError as err:
            print(f"Failed to parse DataLink XML: {err} ({datalink_url})")
            continue

        fits_url = None
        for td in root.iter("{http://www.ivoa.net/xml/VOTable/v1.3}TD"):
            text = td.text
            if text and "dataPortal/file" in text:
                fits_url = text
                break

        if not fits_url:
            print("No FITS URL found in DataLink!")
            continue

        print("FITS URL:", fits_url)

        # Step 3: download FITS
        fname = f"{target_name}_{file_index}.fits"

        try:
            r = requests.get(fits_url, auth=(USER, PASS), stream=True)
            r.raise_for_status()
        except requests.HTTPError as err:
            if err.response is not None and err.response.status_code == 401:
                print(f"401 Unauthorized Error while downloading FITS: Check your ESO credentials. ({fits_url})")
                break
            else:
                print(f"Failed to download FITS file: {err} ({fits_url})")
                continue
        except Exception as err:
            print(f"Unexpected error downloading FITS: {err} ({fits_url})")
            continue

        with open(outdir / fname, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)

        print("Saved:", fname)
        file_index += 1

In [ ]:
download_data("FEROS", "HIP10234", "fgk_spectra")

DATALINK: http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2016-09-26T07:15:37.315
FITS URL: https://dataportal.eso.org/dataPortal/file/ADP.2016-09-26T07:15:37.315
Saved: HIP10234_0.fits


In [28]:
from tqdm import tqdm

for star in tqdm(fgk.to_records()):
    download_data(star['origin'], star['star'], "fgk_spectra")
    if star['star_alt1'] != '-':
        download_data(star['origin'], star['star_alt1'], "fgk_spectra")
    if star['star_alt2'] != '-':
        download_data(star['origin'], star['star_alt2'], "fgk_spectra")

  0%|          | 0/521 [00:00<?, ?it/s]

DATALINK: http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2014-09-17T11:19:46.597
FITS URL: https://dataportal.eso.org/dataPortal/file/ADP.2014-09-17T11:19:46.597
Saved: HD195564_0.fits
DATALINK: http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2014-09-24T09:42:26.347
FITS URL: https://dataportal.eso.org/dataPortal/file/ADP.2014-09-24T09:42:26.347
Saved: HD195564_1.fits
DATALINK: http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2014-09-24T09:42:52.327
FITS URL: https://dataportal.eso.org/dataPortal/file/ADP.2014-09-24T09:42:52.327
Saved: HD195564_2.fits
DATALINK: http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2014-09-24T09:42:57.420
FITS URL: https://dataportal.eso.org/dataPortal/file/ADP.2014-09-24T09:42:57.420
Saved: HD195564_3.fits
DATALINK: http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2014-09-24T09:43:15.683
FITS URL: https://dataportal.eso.org/dataPortal/file/ADP.2014-09-24T09:43:15.683
Saved: HD195564_4.fits


  0%|          | 0/521 [02:42<?, ?it/s]


KeyboardInterrupt: 